# Melanoma Detection — HAM10000 Training (Kaggle)

Train a binary melanoma classifier using transfer learning on ResNet18.

**Before running:**
1. Enable GPU: Settings → Accelerator → GPU T4 x2
2. Add dataset: `kmader/skin-cancer-mnist-ham10000`

**Output:** `melanoma_model.pth` — download and place in `models/` folder of your project.

In [ ]:
import os
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

DATA_DIR = Path('/kaggle/input/skin-cancer-mnist-ham10000')
METADATA_PATH = DATA_DIR / 'HAM10000_metadata.csv'
IMAGE_DIRS = [DATA_DIR / 'HAM10000_images_part_1', DATA_DIR / 'HAM10000_images_part_2']
CLASS_NAMES = ['benign', 'melanoma']
BATCH_SIZE = 32
NUM_EPOCHS_STAGE1 = 10
NUM_EPOCHS_STAGE2 = 15
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
df = pd.read_csv(METADATA_PATH)
df['label'] = df['dx'].apply(lambda x: 1 if x == 'mel' else 0)
df['label_name'] = df['label'].map({0: 'benign', 1: 'melanoma'})

image_id_to_path = {}
for image_dir in IMAGE_DIRS:
    for image_path in image_dir.glob('*.jpg'):
        image_id_to_path[image_path.stem] = image_path

df['image_path'] = df['image_id'].map(image_id_to_path)
df = df.dropna(subset=['image_path']).reset_index(drop=True)

print(f'Total images: {len(df)}')
print(df['label_name'].value_counts())
print(f'Melanoma ratio: {df["label"].mean():.2%}')

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=RANDOM_SEED,
)

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class Ham10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = int(row['label'])
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = Ham10000Dataset(train_df, transform=train_transform)
val_dataset = Ham10000Dataset(val_df, transform=val_transform)

class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_df['label'].values]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
def build_model(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model


def set_backbone_trainable(model, trainable: bool):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True
    if trainable:
        for param in model.parameters():
            param.requires_grad = True


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, num_epochs, lr, freeze_backbone=True):
    set_backbone_trainable(model, trainable=not freeze_backbone)
    model = model.to(DEVICE)

    weights = torch.tensor([1.0, len(train_df) / max(1, train_df['label'].sum())], dtype=torch.float32)
    weights = weights / weights.sum() * 2
    criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE))
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f'Epoch {epoch + 1}/{num_epochs} | '
            f'train loss {train_loss:.4f} acc {train_acc:.4f} | '
            f'val loss {val_loss:.4f} acc {val_acc:.4f}'
        )

    model.load_state_dict(best_state)
    return model, history

In [ ]:
print('Stage 1: Train classification head (frozen backbone)')
model = build_model()
model, history_stage1 = train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=NUM_EPOCHS_STAGE1,
    lr=1e-3,
    freeze_backbone=True,
)

print('\nStage 2: Fine-tune entire network')
model, history_stage2 = train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=NUM_EPOCHS_STAGE2,
    lr=1e-4,
    freeze_backbone=False,
)

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
print('Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))

melanoma_recall = np.sum((np.array(all_labels) == 1) & (np.array(all_preds) == 1)) / max(1, np.sum(np.array(all_labels) == 1))
print(f'\nMelanoma recall (most important metric): {melanoma_recall:.2%}')

In [ ]:
output_path = Path('/kaggle/working/melanoma_model.pth')
checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'architecture': 'resnet18',
    'num_classes': 2,
}
torch.save(checkpoint, output_path)
print(f'Model saved to {output_path}')
print('Download this file and place it in models/melanoma_model.pth in your project.')